# Testing environment for training a Neural Network for Localization


In [1]:
import os

import numpy as np

from scripts.data_loader import load_dataframe

# source files
BASE_DIR = "data/"
FULL_DATA_SET = "Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat"
filename = os.path.join(BASE_DIR, FULL_DATA_SET)

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename)

Loaded dataframe from .h5 file: data/Campaign_data_NBIoT_1_2_3_4_5_6_interpolated_smoothed.mat_dataframe.h5


In [2]:
from scripts.utils import RF_PARAM
from scripts.utils import extract_unique_npcis
from scripts.weighted_coverage import create_point_matrix

# Experiement setup for the ANN
rf_param = RF_PARAM.NSINR
operator_choice = np.array([1, 10, 88])
unique_npcis = extract_unique_npcis(df, operator_choice)
n_runs = 40
results = []

r, _ = create_point_matrix(df.sample(1), unique_npcis, rf_param)

r

array([[-40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -17.4675, -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    ,  25.63  , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -13.47  , -40.    ,
        -40.    , -40.    , -40.    , -40.    ,  21.495 , -40.    ,
         26.435 , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , -40.    , -40.    ,
        -40.    , -40.    , -40.    , -40.    , 

## Prepare the data for model training

In [23]:

from scripts.weighted_coverage import create_point_matrix
from scripts.utils import extract_unique_npcis, haversine_distance
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import LearningRateScheduler
from tensorflow.keras import Input
from pyproj import CRS, Transformer

# Constants
INITIAL_LR = 0.1
BATCH_SIZE = 48
EPOCHS = 60

# Coordinate system transformations
wgs84 = CRS.from_epsg(4326)
utm = CRS.from_epsg(32633)  # Example UTM zone 33N
transformer_to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
transformer_to_wgs84 = Transformer.from_crs(utm, wgs84, always_xy=True)


def transform_to_utm(coords: np.array) -> np.array:
    """Convert WGS84 coordinates to UTM."""
    return np.array([transformer_to_utm.transform(lng, lat) for lat, lng in coords])


def transform_to_wgs84(coords: np.array) -> np.array:
    """Convert UTM coordinates to WGS84."""
    return np.array([transformer_to_wgs84.transform(lat, lng) for lat, lng in coords])


def prepare_data(df: pd.DataFrame, unique_npcis: np.array, rf_param, test_size: float = 0.3, random_seed: int = 42):
    """
    Prepare the dataset for model training and testing.
    
    :param df: Input DataFrame.
    :param unique_npcis: Unique NPCI identifiers.
    :param rf_param: RF parameter.
    :param test_size: Fraction of data for testing.
    :param random_seed: Random seed for reproducibility.
    :return: Tuple of train/test data and scalers.
    """
    # Shuffle the dataframe
    df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    # Get the point matrix (row of NSINR values for each unique <NPCI, eNodeB-ID, operatorID> triplet
    point_matrix, _ = create_point_matrix(df, unique_npcis, rf_param)

    X = point_matrix
    y = df[['lat', 'lng']].values

    # Transform coordinates to UTM
    y_utm = transform_to_utm(y)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y_utm, test_size=test_size, random_state=random_seed)

    # Min-Max normalization
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)
    y_train = scaler_y.fit_transform(y_train)
    y_test = scaler_y.transform(y_test)

    return X_train, X_test, y_train, y_test, scaler_X, scaler_y


def build_model(input_shape):
    model = keras.Sequential([
        Input(shape=input_shape, name='input_layer'),
        Dense(600, activation='relu', name='dense_600'),
        BatchNormalization(name='batch_norm_600'),
        Dense(300, activation='relu', name='dense_300'),
        BatchNormalization(name='batch_norm_300'),
        Dense(150, activation='relu', name='dense_150'),
        BatchNormalization(name='batch_norm_150'),
        Dropout(0.2, name='dropout'),
        Dense(2, name='output_layer')
    ])
    return model


def train_model(X_train, y_train):
    """
    Train the Neural Network.
    
    :param X_train: Training features.
    :param y_train: Training labels.
    :return: Trained Keras model.
    """
    input_shape = (X_train.shape[1],)
    # model = keras.Sequential([
    #     Input(shape=input_shape, name='input_layer'),
    #     Dense(600, activation='relu', name='dense_600'),
    #     BatchNormalization(name='batch_norm_600'),
    #     Dense(300, activation='relu', name='dense_300'),
    #     BatchNormalization(name='batch_norm_300'),
    #     Dense(150, activation='relu', name='dense_150'),
    #     BatchNormalization(name='batch_norm_150'),
    #     Dropout(0.2, name='dropout'),
    #     Dense(2, name='output_layer')
    # ])

    model = build_model(input_shape)

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
                  loss='mean_squared_error',
                  metrics=['mae'])

    def scheduler(epoch, lr):
        return lr * 0.1 if epoch % 20 == 0 and epoch else lr

    lr_scheduler = LearningRateScheduler(scheduler)

    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, callbacks=[lr_scheduler],
              verbose=0)

    return model


def model_predict(model, X_test, scaler_y):
    """
    Predict locations using the trained model and return positions in WGS84 format.
    
    :param model: Trained Keras model.
    :param X_test: Test features.
    :param scaler_y: Scaler for inverse transformation.
    :return: Predicted positions in WGS84.
    """
    y_pred = model.predict(X_test, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred)
    y_pred_wgs84 = transform_to_wgs84(y_pred)

    return y_pred_wgs84




In [24]:
from scripts.utils import RF_PARAM

# Experiement setup for the ANN
rf_param = RF_PARAM.NSINR
operator_choice = np.array([1, 10, 88])
unique_npcis = extract_unique_npcis(df, operator_choice)
n_runs = 1
results = []

print(f"""
Building, training and testing Neural Network

🧪 Experiment setup 🧪 
⚙️ RF PARAM {rf_param}
📶 Operator choice {operator_choice}
🔁 Number of runs {n_runs}
_________________________________
""")

for i in range(n_runs):
    print(f"\r🔄 {i + 1} / {n_runs}", end="")

    # Prepare the data
    X_train, X_test, y_train, y_test, scaler_X, scaler_y = prepare_data(df, unique_npcis, rf_param,
                                                                        random_seed=random_seeds[i])
    # Time and train the model 
    start = time.time()
    model = train_model(X_train, y_train)
    offline_runtime = time.time() - start

    # Evaluate the model
    test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

    # Time the prediction and transformation time
    start = time.time()
    predicted = model_predict(model, X_test, scaler_y)
    online_runtime = time.time() - start

    # Convert test labels back to WGS84 for comparison
    #y_test_wgs84 = np.array([transformer_to_wgs84.transform(x, y) for x, y in scaler_y.inverse_transform(y_test)])
    y_test_wgs84 = transform_to_wgs84(scaler_y.inverse_transform(y_test))
    # Calculate error
    errors = haversine_distance(y_test_wgs84[:, 0], y_test_wgs84[:, 1], predicted[:, 0], predicted[:, 1])

    results.append(
        (i,
         offline_runtime,
         online_runtime,
         test_loss,
         test_mae,
         errors.mean(),
         errors.std(),
         errors.min(),
         errors.max()
         )
    )

    print(f"\r✅ {i + 1} run completed")

result_df = pd.DataFrame(results,
                         columns=['Run', 'offline_runtime', 'online_runtime', 'test_loss', 'test_mae', 'mean_pos_error',
                                  'std_pos_error', 'min_pos_error', 'max_pos_error'])

print(f"""_________________________________
📊 Result over: \t {n_runs} runs
📏 Mean error: \t\t {result_df['mean_pos_error'].mean():.2f} m
⏱️ Offline runtime:  {result_df['offline_runtime'].mean():.2f} s
⏱️ Online runtime: \t {result_df['online_runtime'].mean():.2f} s
🎯 Test MAE: \t\t {result_df['test_mae'].mean():.2f} units
""")


Building, training and testing Neural Network

🧪 Experiment setup 🧪 
⚙️ RF PARAM RF_PARAM.NSINR
📶 Operator choice [ 1 10 88]
🔁 Number of runs 1
_________________________________

✅ 1 run completed
_________________________________
📊 Result over: 	 1 runs
📏 Mean error: 		 120.64 m
⏱️ Offline runtime:  10.17 s
⏱️ Online runtime: 	 0.10 s
🎯 Test MAE: 		 0.02 units

